In [16]:
import torch
import torch.nn as nn
import random

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Optional speed boost
torch.backends.cudnn.benchmark = True

Using device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [17]:
with open("names.txt", "r", encoding="utf-8") as f:
    names = f.read().splitlines()

names = [name.lower() for name in names]

# Add start/end token
names = ["." + name + "." for name in names]

print("Sample names:", names[:5])

Sample names: ['.emma.', '.olivia.', '.ava.', '.isabella.', '.sophia.']


In [18]:
chars = sorted(list(set("".join(names))))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

vocab_size = len(chars)

print("Vocab:", chars)
print("Vocab size:", vocab_size)

Vocab: ['.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocab size: 27


In [19]:
def encode(s):
    return [stoi[c] for c in s]

def decode(indices):
    return ''.join([itos[i] for i in indices])

In [20]:
X, Y = [], []

for name in names:
    encoded = encode(name)
    X.append(torch.tensor(encoded[:-1], dtype=torch.long))
    Y.append(torch.tensor(encoded[1:], dtype=torch.long))

print("Total samples:", len(X))

Total samples: 32033


In [21]:
class NameDataset(Dataset):
    def __init__(self, X, Y):
        # X and Y are already tensors from the preprocessing cell
        self.X = [x.clone().detach() for x in X]
        self.Y = [y.clone().detach() for y in Y]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


def collate_fn(batch):
    X_batch, Y_batch = zip(*batch)

    X_padded = pad_sequence(X_batch, batch_first=True, padding_value=0)
    Y_padded = pad_sequence(Y_batch, batch_first=True, padding_value=-100)

    return X_padded, Y_padded


dataset = NameDataset(X, Y)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    collate_fn=collate_fn
)


In [22]:
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = self.fc(out)   
        return out


model = CharLSTM(vocab_size).to(device)
print(model)

CharLSTM(
  (embedding): Embedding(27, 64, padding_idx=0)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=27, bias=True)
)


In [23]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

In [26]:
import time

epochs = 100

start_time = time.time()

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for X_batch, Y_batch in loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        logits = model(X_batch)   # (B, T, vocab)

        loss = loss_fn(
            logits.view(-1, vocab_size),
            Y_batch.view(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

print("Total training time:", time.time() - start_time)


Epoch 1/100, Loss: 1.7927
Epoch 2/100, Loss: 1.7894
Epoch 3/100, Loss: 1.7871
Epoch 4/100, Loss: 1.7837
Epoch 5/100, Loss: 1.7817
Epoch 6/100, Loss: 1.7800
Epoch 7/100, Loss: 1.7777
Epoch 8/100, Loss: 1.7734
Epoch 9/100, Loss: 1.7727
Epoch 10/100, Loss: 1.7711
Epoch 11/100, Loss: 1.7675
Epoch 12/100, Loss: 1.7653
Epoch 13/100, Loss: 1.7634
Epoch 14/100, Loss: 1.7623
Epoch 15/100, Loss: 1.7600
Epoch 16/100, Loss: 1.7574
Epoch 17/100, Loss: 1.7563
Epoch 18/100, Loss: 1.7540
Epoch 19/100, Loss: 1.7517
Epoch 20/100, Loss: 1.7512
Epoch 21/100, Loss: 1.7487
Epoch 22/100, Loss: 1.7459
Epoch 23/100, Loss: 1.7441
Epoch 24/100, Loss: 1.7432
Epoch 25/100, Loss: 1.7420
Epoch 26/100, Loss: 1.7411
Epoch 27/100, Loss: 1.7375
Epoch 28/100, Loss: 1.7376
Epoch 29/100, Loss: 1.7351
Epoch 30/100, Loss: 1.7341
Epoch 31/100, Loss: 1.7320
Epoch 32/100, Loss: 1.7313
Epoch 33/100, Loss: 1.7293
Epoch 34/100, Loss: 1.7283
Epoch 35/100, Loss: 1.7258
Epoch 36/100, Loss: 1.7254
Epoch 37/100, Loss: 1.7223
Epoch 38/1

In [27]:
def generate_name(model, max_len=20, temperature=1.0):
    model.eval()

    idx = stoi["."]
    result = [idx]

    for _ in range(max_len):
        x = torch.tensor(result, dtype=torch.long).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

        next_token_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_token_logits, dim=-1).squeeze(0)

        idx = torch.multinomial(probs, num_samples=1).item()

        if itos[idx] == ".":
            break

        result.append(idx)

    return decode(result[1:])


In [28]:
print("\nGenerated Names:\n" + "="*30)

for _ in range(20):
    print(generate_name(model, temperature=0.8))


Generated Names:
henrick
zayna
dawaya
shriansh
suha
everlynn
karima
cate
timber
aedon
abhisrath
taven
khelia
mayana
delina
referin
maleeah
elia
haydar
keaton
